> **Sentetik veri.** Bu defter `python -m sentetik.uret` ile üretilen uydurma veriyle çalışır. Sayılar gerçek bir şirkete ait değildir ve gerçek projenin sonuçlarını temsil etmez; amaç yöntemi uçtan uca göstermektir. *(Synthetic data: numbers are not real results.)*

# Sinyal — 02 · Bazlar ve Deney (geliştirme)

**Protokol (tanimlar v1.1):**
- Bu defter **yalnızca kayan pencere doğrulamasını** kullanır: doğrulama kesimleri 2025-04, 05, 06, 07; her birinde eğitim, etiket penceresi o kesimden önce kapanan eğitim kesimleridir.
- **Test (2025-10-01) ve test2 (2026-01-01) bu defterde hiç kullanılmaz.** Onlar, bütün seçimler dondurulduktan sonra `03_test.ipynb` defterinde bir kez açılacak.
- Bu defterde verilen kararlar: (1) davranış dönüşümü (yüzdelik / z-skor / ham), (2) ana model (lojistik / LightGBM). Kararlar bir çalışma günlüğüne tarihli olarak yazılır.

**Kollar:** A (sipariş), A + bayrak, B (sipariş + davranış) · **Modeller:** lojistik regresyon, LightGBM · **Bazlar:** kural (yakınlık), RFM, BG/NBD.

In [ ]:
import sys, time
from pathlib import Path

KOK = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(KOK))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.veri import tablo_yukle
from src.ozellikler import (hazirla, model_a_ozellikleri, model_a_bayrak_ozellikleri,
                            model_b_ozellikleri)
from src.degerlendirme import (kayan_katlar, metrikler, isaret_tutarli, ISARET_MIN_KAT,
                                 alt_kumeler)
from src.modeller import baz_kural, baz_rfm, baz_bgnbd, egit_ve_skorla

HIZLI = True           # önce True (eğitim satırlarının %30'u, sonuçlar raporlanmaz), sonra False ile tam
VARSAYILAN_DONUSUM = "yuzdelik_aktif"   # tanimlar v1.2
SONUC = KOK / "docs" / "sonuclar"; SONUC.mkdir(parents=True, exist_ok=True)
FIG = KOK / "notebooks" / "figurler"; FIG.mkdir(parents=True, exist_ok=True)
pd.set_option("display.float_format", "{:.4f}".format)

a = tablo_yukle("egitim_a")
b = tablo_yukle("egitim_b")
df = hazirla(a, b, davranis_yontemi=VARSAYILAN_DONUSUM)
df["y_etiket"] = df["y_etiket"].astype(int)
assert not df.loc[df["kume"].isin(["test", "test2"])].index.isin(
    [i for _, e, v in kayan_katlar(df) for i in list(e) + list(v)]).any(), "test satırı geliştirmeye karıştı!"

SETLER = {
    "A":        model_a_ozellikleri(df),
    "A+bayrak": model_a_bayrak_ozellikleri(df),
    "B":        model_b_ozellikleri(df),
}
{k: len(v) for k, v in SETLER.items()}

## MLflow (isteğe bağlı)
Kayıt deposu: `~/mlruns_sinyal/mlflow.db`. Arayüz: `mlflow ui --backend-store-uri sqlite:///$HOME/mlruns_sinyal/mlflow.db`. MLflow kurulu değilse bu bölüm kendini atlar.

In [ ]:
try:
    import mlflow
    depo = Path.home() / "mlruns_sinyal"; depo.mkdir(exist_ok=True)
    mlflow.set_tracking_uri(f"sqlite:///{(depo / 'mlflow.db').as_posix()}")
    mlflow.set_experiment("sinyal_02_gelistirme")
    MLFLOW = not HIZLI
except Exception as hata:
    print("MLflow devre dışı:", hata); MLFLOW = False

def kaydet(satir):
    if not MLFLOW:
        return
    with mlflow.start_run(run_name=f"{satir['model']}|{satir['set']}|{satir['kat']}"):
        mlflow.log_params({k: satir[k] for k in ["model", "set", "kat", "alt_kume"]})
        mlflow.log_metrics({k.replace('@', '_at_'): v for k, v in satir.items()
                            if isinstance(v, (int, float)) and k not in ["n"] and pd.notna(v)})

## Değerlendirme yardımcıları
Her skor yedi alt kümede ölçülür (`src.degerlendirme.alt_kumeler`): **tum** · **karar** (son siparişi 30 günden eski) · **kapsam** / **davranissiz** · **tek_siparis** / **tekrar** · **uzak_180** (son siparişi 180 günden eski). ROC-AUC yalnızca raporlanır.

In [ ]:
# alt_kumeler artık src/degerlendirme.py'de (7 alt küme; 02_gelisim.csv ilk çalıştırmada 4 alt küme ile üretildi)
sonuclar, tahminler = [], {}

def degerlendir(model, set_adi, kat, d, skor, olasilik):
    skor = np.asarray(skor, dtype=float)
    tahminler[(model, set_adi, kat)] = skor
    for ak, maske in alt_kumeler(d).items():
        satir = {"model": model, "set": set_adi, "kat": kat.date().isoformat(), "alt_kume": ak,
                 **metrikler(d["y_etiket"].values[maske], skor[maske], olasilik)}
        sonuclar.append(satir); kaydet(satir)

## 1. Bazlar

In [ ]:
for v, egt, val in kayan_katlar(df):
    d = df.loc[val]
    degerlendir("kural", "-", v, d, baz_kural(d), olasilik=False)
    degerlendir("rfm",   "-", v, d, baz_rfm(d),   olasilik=False)
    try:
        t0 = time.time()
        degerlendir("bgnbd", "-", v, d, baz_bgnbd(d), olasilik=False)
        print(v.date(), "BG/NBD", round(time.time() - t0), "sn")
    except Exception as hata:
        print("BG/NBD atlandı:", hata)

bazlar = pd.DataFrame(sonuclar).query("alt_kume == 'tum'")
bazlar.pivot_table(index="model", columns="kat", values=["capture@10", "pr_auc"]).round(4)

## 2. Deney: A, A + bayrak, B × lojistik, LightGBM
Toplam 4 kat × 3 set × 2 model = 24 eğitim. Tam veride ~20–40 dakika sürebilir (`HIZLI=True` ile önce deneyebilirsin).

In [ ]:
for v, egt, val in kayan_katlar(df):
    e, d = df.loc[egt], df.loc[val]
    if HIZLI:
        e = e.sample(frac=0.3, random_state=42)
    for set_adi, kolonlar in SETLER.items():
        for model in ["lojistik", "lightgbm"]:
            t0 = time.time()
            p = egit_ve_skorla(model, e, d, kolonlar)
            degerlendir(model, set_adi, v, d, p, olasilik=True)
            print(f"{v.date()} {model:9s} {set_adi:9s} {time.time() - t0:5.0f} sn")

## 3. Özet tablolar (kat ortalaması)

In [ ]:
S = pd.DataFrame(sonuclar)
S.to_csv(SONUC / ("02_gelisim_HIZLI.csv" if HIZLI else "02_gelisim.csv"), index=False)

ozet = (S.groupby(["alt_kume", "model", "set"])[["pr_auc", "capture@10", "kesinlik@10", "brier", "roc_auc"]]
          .mean().round(4))
ozet.loc["tum"]

In [ ]:
# Diğer alt kümeler
for ak in ["karar", "kapsam", "davranissiz", "tek_siparis", "tekrar", "uzak_180"]:
    print(f"\n=== {ak} ===")
    display(ozet.loc[ak])

## 4. Deneyin özü: B − A ve (A+bayrak) − A farkı, kat kat
Kilitli ölçüt (test için): fark ≥ 1 yüzde puan ve kayan pencerelerin en az 3'te 2'sinde aynı işaret. 4 katta bu **en az 3 kat** demek (tanimlar v1.2 açıklaması); işaret, test farkının işaretiyle karşılaştırılır. Burada yalnızca **geliştirme göstergesi** olarak bakıyoruz.

In [ ]:
def fark_tablosu(metrik="capture@10"):
    t = S.pivot_table(index=["alt_kume", "model", "kat"], columns="set", values=metrik)
    t = t[["A", "A+bayrak", "B"]]
    t["B-A"] = t["B"] - t["A"]
    t["bayrak-A"] = t["A+bayrak"] - t["A"]
    t["B-bayrak"] = t["B"] - t["A+bayrak"]
    return t

F = fark_tablosu("capture@10")
tutarlilik = (F.groupby(level=["alt_kume", "model"])
                .agg(B_A_ort=("B-A", "mean"),
                     B_A_pozitif_kat=("B-A", lambda x: int((x > 0).sum())),
                     B_A_negatif_kat=("B-A", lambda x: int((x < 0).sum())),
                     bayrak_A_ort=("bayrak-A", "mean"), B_bayrak_ort=("B-bayrak", "mean")))
tutarlilik[f"isaret_tutarli_{ISARET_MIN_KAT}/4"] = (
    F.groupby(level=["alt_kume", "model"])["B-A"].apply(isaret_tutarli))   # tanimlar v1.2, madde 2
for c in ["B_A_ort", "bayrak_A_ort", "B_bayrak_ort"]:
    tutarlilik[c] = (tutarlilik[c] * 100).round(2)   # yüzde puan
tutarlilik

## 5. Davranış dönüşümü seçimi (yalnızca B, LightGBM ve lojistik)
yüzdelik_aktif (varsayılan, v1.2) · yüzdelik (v1.1) · z-skor · ham. Seçim kat ortalaması capture@10'a göre; fark < 0,3 puan ise varsayılan korunur. Varsayılanın sonuçları bölüm 2'den alınır (yeniden eğitilmez).

In [ ]:
# Varsayılan dönüşüm bölüm 2'de zaten eğitildi: sonuçları oradan al
donusum = (S.query("set == 'B' and alt_kume == 'tum'")
             .drop(columns=["set", "alt_kume"]).assign(yontem=VARSAYILAN_DONUSUM)
             .to_dict("records"))
for yontem in [y for y in ["yuzdelik_aktif", "yuzdelik", "zskor", "ham"] if y != VARSAYILAN_DONUSUM]:
    dfy = hazirla(a, b, davranis_yontemi=yontem); dfy["y_etiket"] = dfy["y_etiket"].astype(int)
    kol = model_b_ozellikleri(dfy)
    for v, egt, val in kayan_katlar(dfy):
        e, d = dfy.loc[egt], dfy.loc[val]
        if HIZLI:
            e = e.sample(frac=0.3, random_state=42)
        for model in ["lojistik", "lightgbm"]:
            p = egit_ve_skorla(model, e, d, kol)
            m = metrikler(d["y_etiket"], p)
            donusum.append({"yontem": yontem, "model": model, "kat": v.date().isoformat(), **m})
            print(yontem, model, v.date(), round(m["capture@10"], 4))
    del dfy

D = pd.DataFrame(donusum)
D.to_csv(SONUC / ("02_donusum_HIZLI.csv" if HIZLI else "02_donusum.csv"), index=False)
D.groupby(["model", "yontem"])[["pr_auc", "capture@10"]].mean().round(4)

## 6. Kararlar (doldurulacak)
- Davranış dönüşümü: … (varsayılan yuzdelik_aktif; başka biri seçilirse tanimlar v1.3)
- Ana model: …
- Gözlemler: …